# ДЗ 5

Марковский Артём Владиславович, 466636, J3310

Репозиторий: https://github.com/shlaapnik/EngData/tree/kubernetes/Kubernetes

In [1]:
import csv
import io
import json
import math
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

PROJECT = Path.cwd()
K8S = PROJECT / "k8s"
JOBS = K8S / "jobs"
LOGS = PROJECT / "logs"
JOBS.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(exist_ok=True)
OUTSIDE = "host.docker.internal:9000"
INSIDE = "rustfs.hw5.svc:9000"

sys.path.insert(0, str(PROJECT.parent / "MapReduce"))
import common

s3 = common.s3_client()

print("python", sys.version.split()[0])

python 3.13.7


In [2]:
JOB = """apiVersion: batch/v1
kind: Job
metadata:
  name: {name}
  namespace: {namespace}
spec:
  completions: {parts}
  parallelism: {parallel}
  completionMode: Indexed
  backoffLimit: {backoff}
  template:
    metadata:
      labels:
        app: pairs
    spec:
      restartPolicy: Never
{priority}      containers:
        - name: pairs
          image: pairs-job:hw5
          imagePullPolicy: Never
          args: [{args}]
          env:
            - {{name: S3_ENDPOINT, value: "{endpoint}"}}
            - {{name: WORKERS, value: "4"}}
          resources:
            requests: {{cpu: "{cpu}", memory: {memory}}}
            limits: {{cpu: "{cpu}", memory: {memory}}}
"""


def quiet(*arguments: str) -> str:
    result = subprocess.run(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                            errors="replace", timeout=600)

    return result.stdout.strip()


def sh(*arguments: str) -> None:
    print("$ " + " ".join(arguments) + "\n" + quiet(*arguments) + "\n")


def job(name: str, args: list[str], memory: str, endpoint: str, cpu: str = "4", parts: int = 1, parallel: int = 1,
        namespace: str = "hw5", priority: str = "", backoff: int = 3) -> None:
    priority_line = "      priorityClassName: " + priority + "\n" if priority else ""
    text = JOB.format(name=name, namespace=namespace, parts=parts, parallel=parallel, backoff=backoff, priority=priority_line,
                      args=", ".join(args), endpoint=endpoint, cpu=cpu, memory=memory)
    path = JOBS / (name + ".yaml")
    path.write_text(text, encoding="utf-8")
    quiet("kubectl", "delete", "job", name, "-n", namespace, "--ignore-not-found")
    sh("kubectl", "apply", "-f", path.relative_to(PROJECT).as_posix())


def job_state(name: str, namespace: str = "hw5") -> str:
    status = json.loads(quiet("kubectl", "get", "job", name, "-n", namespace, "-o", "json")).get("status", {})
    finished = [condition["type"] for condition in status.get("conditions", [])
                if condition["type"] in ("Complete", "Failed") and condition["status"] == "True"]

    return finished[0] if finished else ""


def wait_job(name: str, namespace: str = "hw5", timeout: float = 3600) -> str:
    deadline = time.monotonic() + timeout

    while time.monotonic() < deadline:
        state = job_state(name, namespace)

        if state:
            return state

        time.sleep(2)

    raise TimeoutError("задание " + name + " не закончилось за " + str(timeout) + " с")


def pods(name: str, namespace: str = "hw5") -> list[dict]:
    return json.loads(quiet("kubectl", "get", "pods", "-n", namespace, "-l", "job-name=" + name, "-o", "json"))["items"]


def wait_pods(name: str, check, namespace: str = "hw5", timeout: float = 600) -> None:
    deadline = time.monotonic() + timeout

    while time.monotonic() < deadline:
        if check([pod["status"]["phase"] for pod in pods(name, namespace)]):
            return

        time.sleep(1)

    raise TimeoutError("поды " + name + " не дошли до нужного состояния")


def moment(text: str) -> datetime:
    return datetime.fromisoformat(text.replace("Z", "+00:00"))


def timeline(name: str, namespace: str = "hw5") -> None:
    print("под, часть, ждал планирования с, работал с, итог")

    for pod in sorted(pods(name, namespace), key=lambda item: item["metadata"]["creationTimestamp"]):
        created = moment(pod["metadata"]["creationTimestamp"])
        scheduled = [condition for condition in pod["status"].get("conditions", []) if condition["type"] == "PodScheduled"]
        waited = round((moment(scheduled[0]["lastTransitionTime"]) - created).total_seconds()) if scheduled else "-"
        state = (pod["status"].get("containerStatuses") or [{}])[0].get("state", {})
        finished = state.get("terminated")
        worked = round((moment(finished["finishedAt"]) - moment(finished["startedAt"])).total_seconds()) if finished else "-"
        result = finished["reason"] + " " + str(finished["exitCode"]) if finished else pod["status"]["phase"]
        print(pod["metadata"]["name"], pod["metadata"]["annotations"].get("batch.kubernetes.io/job-completion-index"),
              waited, worked, result)


def save_logs(name: str, namespace: str = "hw5") -> None:
    text = quiet("kubectl", "logs", "-n", namespace, "-l", "job-name=" + name, "--prefix", "--tail=-1",
                 "--max-log-requests=20")
    (LOGS / (name + ".log")).write_text(text + "\n", encoding="utf-8")
    print("logs/" + name + ".log")
    print(text + "\n")


def save_events(name: str, namespace: str = "hw5") -> None:
    events = json.loads(quiet("kubectl", "get", "events", "-n", namespace, "-o", "json"))["items"]
    lines = []

    for event in sorted(events, key=lambda item: item.get("lastTimestamp") or item.get("eventTime") or ""):
        target = event["involvedObject"]["name"]

        if target.startswith(name):
            lines.append((event.get("lastTimestamp") or event.get("eventTime") or "")[11:19] + " " + event["type"] + " "
                         + event["reason"] + " " + event["involvedObject"]["kind"] + "/" + target + ": " + event["message"])

    (LOGS / (name + ".events.log")).write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("logs/" + name + ".events.log")
    print("\n".join(lines) + "\n")


def fresh_rustfs() -> None:
    sh("kubectl", "rollout", "restart", "deployment/rustfs", "-n", "hw5")
    sh("kubectl", "rollout", "status", "deployment/rustfs", "-n", "hw5", "--timeout=300s")

    for attempt in range(90):
        try:
            s3.head_object(Bucket=common.BUCKET, Key=common.CSV_KEY)
            break
        except Exception:
            time.sleep(2)

    time.sleep(5)

## 1. Кластер

Kubernetes из Docker Desktop в режиме kubeadm, один узел, это виртуальная машина Docker с 16 ядрами и 7.3 ГиБ памяти, kubectl идёт вместе с Docker Desktop

In [3]:
sh("kubectl", "version")
sh("kubectl", "get", "nodes", "-o", "wide")
sh("kubectl", "get", "node", "docker-desktop", "-o", "jsonpath={.status.capacity}{'\\n'}{.status.allocatable}")
sh("kubectl", "get", "pods", "-n", "kube-system")

$ kubectl version
Client Version: v1.34.1
Kustomize Version: v5.7.1
Server Version: v1.34.1

$ kubectl get nodes -o wide
NAME             STATUS   ROLES           AGE     VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE         KERNEL-VERSION                     CONTAINER-RUNTIME
docker-desktop   Ready    control-plane   7h10m   v1.34.1   192.168.65.3   <none>        Docker Desktop   6.6.87.2-microsoft-standard-WSL2   docker://28.5.1



$ kubectl get node docker-desktop -o jsonpath={.status.capacity}{'\n'}{.status.allocatable}
{"cpu":"16","ephemeral-storage":"1055762868Ki","hugepages-1Gi":"0","hugepages-2Mi":"0","memory":"7704768Ki","pods":"110"}
{"cpu":"16","ephemeral-storage":"972991057538","hugepages-1Gi":"0","hugepages-2Mi":"0","memory":"7602368Ki","pods":"110"}

$ kubectl get pods -n kube-system
NAME                                     READY   STATUS    RESTARTS       AGE
coredns-66bc5c9577-bhr78                 1/1     Running   0              7h10m
coredns-66bc5c9577-qnw4t                 1/1     Running   0              7h10m
etcd-docker-desktop                      1/1     Running   0              7h10m
kube-apiserver-docker-desktop            1/1     Running   0              7h10m
kube-controller-manager-docker-desktop   1/1     Running   0              7h10m
kube-proxy-4mr8q                         1/1     Running   0              7h10m
kube-scheduler-docker-desktop            1/1     Running   0           

## 2. Образ

В образ кладу pairs_job.py, а map и reduce он берёт из файлов ДЗ 3 mapreduce_job.py и common.py. Программа считает пары навыков в одной части файла, 373 сплита по 64 МиБ делятся на 8 частей по 47, часть считают 4 процесса-воркера, результат части уходит в S3. Режим reduce складывает части, пишет итоговые пары и сравнивает топ-10 с results/mapreduce/skills_k2.csv из ДЗ 3, в конце программа печатает пик памяти своего контейнера из cgroup, файл memory.peak

In [4]:
print((PROJECT / "Dockerfile").read_text(encoding="utf-8"))
sh("docker", "build", "-q", "-f", "Dockerfile", "-t", "pairs-job:hw5", "..")
sh("docker", "image", "ls", "pairs-job")

FROM python:3.13.7-slim

ENV PYTHONUNBUFFERED=1
WORKDIR /app
RUN pip install --no-cache-dir boto3==1.43.118 pyarrow==26.0.0 tqdm==4.70.1
COPY MapReduce/common.py MapReduce/mapreduce_job.py Kubernetes/pairs_job.py ./

ENTRYPOINT ["python", "pairs_job.py"]



$ docker build -q -f Dockerfile -t pairs-job:hw5 ..
sha256:2985176eae1b5355b7c1de739cf54dda3c23e907a1ebb452d44a025faad0a78e



$ docker image ls pairs-job
REPOSITORY   TAG       IMAGE ID       CREATED       SIZE
pairs-job    hw5       2985176eae1b   7 hours ago   474MB



## 3. Задание с заявками и пределами

Пока S3 снаружи кластера, RustFS из ДЗ 3 работает в обычном контейнере Docker, под ходит к нему через host.docker.internal:9000. Задание считает часть 0, заявки равны пределам, 4 ядра, сначала даю 256 МиБ памяти

In [5]:
sh("docker", "start", "rustfs")

for attempt in range(60):
    try:
        s3.head_object(Bucket=common.BUCKET, Key=common.CSV_KEY)
        break
    except Exception:
        time.sleep(2)

sh("kubectl", "apply", "-f", "k8s/namespace.yaml")
job("pairs-oom", ["part", "oom"], "256Mi", OUTSIDE, backoff=0)
print("итог задания:", wait_job("pairs-oom"))
sh("kubectl", "get", "pods", "-n", "hw5", "-l", "job-name=pairs-oom")
sh("kubectl", "get", "pods", "-n", "hw5", "-l", "job-name=pairs-oom", "-o",
   "jsonpath={.items[0].status.containerStatuses[0].state.terminated}")
save_logs("pairs-oom")
save_events("pairs-oom")

$ docker start rustfs
rustfs



$ kubectl apply -f k8s/namespace.yaml
namespace/hw5 created



$ kubectl apply -f k8s/jobs/pairs-oom.yaml
job.batch/pairs-oom created



итог задания: Failed
$ kubectl get pods -n hw5 -l job-name=pairs-oom
NAME                READY   STATUS      RESTARTS   AGE
pairs-oom-0-sqv5q   0/1     OOMKilled   0          16s



$ kubectl get pods -n hw5 -l job-name=pairs-oom -o jsonpath={.items[0].status.containerStatuses[0].state.terminated}
{"containerID":"docker://533bbc6bb9371300ecb093345b0bee0d64d4bc9237a868b7bc471ea396dc252a","exitCode":1,"finishedAt":"2026-11-23T19:32:32Z","reason":"OOMKilled","startedAt":"2026-11-23T19:32:21Z"}



logs/pairs-oom.log
[pod/pairs-oom-0-sqv5q/pairs] часть 0 из 8 сплиты 0 - 46 воркеров 4 S3 host.docker.internal:9000
[pod/pairs-oom-0-sqv5q/pairs] Traceback (most recent call last):
[pod/pairs-oom-0-sqv5q/pairs]   File "/app/pairs_job.py", line 135, in <module>
[pod/pairs-oom-0-sqv5q/pairs]     run_part(sys.argv[2], int(sys.argv[3]) if len(sys.argv) > 3 else int(os.environ["JOB_COMPLETION_INDEX"]))
[pod/pairs-oom-0-sqv5q/pairs]     ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[pod/pairs-oom-0-sqv5q/pairs]   File "/app/pairs_job.py", line 71, in run_part
[pod/pairs-oom-0-sqv5q/pairs]     for done, result in enumerate(pool.map(mapreduce_job.map_task, tasks), 1):
[pod/pairs-oom-0-sqv5q/pairs]                         ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[pod/pairs-oom-0-sqv5q/pairs]   File "/usr/local/lib/python3.13/concurrent/futures/process.py", line 617, in _chain_from_iterable_of_lists
[pod/pairs-oom-0-sqv5q/p

Под упал через 11 с со статусом OOM, ядро убило по памяти один из процессов-воркеров, поэтому главный процесс вышел с BrokenProcessPool и кодом 1

Тот же расчёт с пределом 2 ГиБ, чтобы измерить пик

In [6]:
job("pairs-measure", ["part", "measure"], "2Gi", OUTSIDE, backoff=0)
print("итог задания:", wait_job("pairs-measure"))
save_logs("pairs-measure")
measured = (LOGS / "pairs-measure.log").read_text(encoding="utf-8")
peak = int(re.search(r"пик памяти контейнера (\d+) МиБ", measured).group(1))
limit = math.ceil(peak * 1.25 / 32) * 32
print("пик", peak, "МиБ, предел с запасом 25% и округлением до 32 МиБ:", limit, "МиБ")

$ kubectl apply -f k8s/jobs/pairs-measure.yaml
job.batch/pairs-measure created



итог задания: Complete


logs/pairs-measure.log
[pod/pairs-measure-0-nts44/pairs] часть 0 из 8 сплиты 0 - 46 воркеров 4 S3 host.docker.internal:9000
[pod/pairs-measure-0-nts44/pairs] готово сплитов 10 из 47
[pod/pairs-measure-0-nts44/pairs] готово сплитов 20 из 47
[pod/pairs-measure-0-nts44/pairs] готово сплитов 30 из 47
[pod/pairs-measure-0-nts44/pairs] готово сплитов 40 из 47
[pod/pairs-measure-0-nts44/pairs] готово сплитов 47 из 47
[pod/pairs-measure-0-nts44/pairs] строк 1218176 вакансий Go 421 уникальных пар 2428 за 184.2 с, записал k8s/measure/part-0.pickle
[pod/pairs-measure-0-nts44/pairs] пик памяти контейнера 404 МиБ, предел 2048 МиБ

пик 404 МиБ, предел с запасом 25% и округлением до 32 МиБ: 512 МиБ


Часть 0 посчиталась за 184 с, пик 404 МиБ, предел 512 МиБ это пик с запасом 25% на разброс между частями

Задание с подобранным пределом

In [7]:
tuned = str(limit) + "Mi"
job("pairs-tuned", ["part", "tuned"], tuned, OUTSIDE, backoff=0)
print("итог задания:", wait_job("pairs-tuned"))
sh("kubectl", "get", "pods", "-n", "hw5", "-l", "job-name=pairs-tuned")
save_logs("pairs-tuned")

$ kubectl apply -f k8s/jobs/pairs-tuned.yaml
job.batch/pairs-tuned created



итог задания: Complete
$ kubectl get pods -n hw5 -l job-name=pairs-tuned
NAME                  READY   STATUS      RESTARTS   AGE
pairs-tuned-0-td94f   0/1     Completed   0          3m6s



logs/pairs-tuned.log
[pod/pairs-tuned-0-td94f/pairs] часть 0 из 8 сплиты 0 - 46 воркеров 4 S3 host.docker.internal:9000
[pod/pairs-tuned-0-td94f/pairs] готово сплитов 10 из 47
[pod/pairs-tuned-0-td94f/pairs] готово сплитов 20 из 47
[pod/pairs-tuned-0-td94f/pairs] готово сплитов 30 из 47
[pod/pairs-tuned-0-td94f/pairs] готово сплитов 40 из 47
[pod/pairs-tuned-0-td94f/pairs] готово сплитов 47 из 47
[pod/pairs-tuned-0-td94f/pairs] строк 1218176 вакансий Go 421 уникальных пар 2428 за 178.8 с, записал k8s/tuned/part-0.pickle
[pod/pairs-tuned-0-td94f/pairs] пик памяти контейнера 352 МиБ, предел 512 МиБ



С пределом 512 МиБ задание прошло, пик 352 МиБ

## 4. RustFS в кластере

Останавливаю контейнер и поднимаю тот же RustFS как Deployment в кластере, диски с диска C подключаются томами hostPath

In [8]:
sh("docker", "stop", "rustfs")
sh("kubectl", "apply", "-f", "k8s/rustfs.yaml")
sh("kubectl", "rollout", "status", "deployment/rustfs", "-n", "hw5", "--timeout=300s")
sh("kubectl", "get", "pods", "-n", "hw5", "-l", "app=rustfs")
sh("kubectl", "get", "service", "rustfs", "-n", "hw5")

for attempt in range(60):
    try:
        head = s3.head_object(Bucket=common.BUCKET, Key=common.CSV_KEY)
        break
    except Exception:
        time.sleep(2)

print("через localhost:9000 объект", common.CSV_KEY, head["ContentLength"], "байт")
print("результаты ДЗ 3:", [item["Key"] for item in s3.list_objects_v2(Bucket=common.BUCKET, Prefix="results/mapreduce/")["Contents"]])

$ docker stop rustfs
rustfs



$ kubectl apply -f k8s/rustfs.yaml
deployment.apps/rustfs created
service/rustfs created



$ kubectl rollout status deployment/rustfs -n hw5 --timeout=300s
Waiting for deployment "rustfs" rollout to finish: 0 of 1 updated replicas are available...
deployment "rustfs" successfully rolled out

$ kubectl get pods -n hw5 -l app=rustfs
NAME                      READY   STATUS    RESTARTS   AGE
rustfs-7fd57bff49-5w7wn   1/1     Running   0          3s



$ kubectl get service rustfs -n hw5
NAME     TYPE           CLUSTER-IP       EXTERNAL-IP   PORT(S)                         AGE
rustfs   LoadBalancer   10.102.169.241   localhost     9000:30955/TCP,9001:32370/TCP   3s



через localhost:9000 объект raw/vacancies_2024.csv 24970431974 байт
результаты ДЗ 3: ['results/mapreduce/skills_k1.csv', 'results/mapreduce/skills_k2.csv', 'results/mapreduce/skills_k3.csv']


## 5. Параллельный расчёт

Indexed Job из 8 частей, все 8 разрешено запускать сразу, каждая часть 4 ядра и подобранный предел памяти. У меня на компе 16 ядер, около 1 ядра на систему и 2 ядра RustFS, поэтому одновременно влезают 3 части, остальные ждут в Pending, пока не освободится процессор. Перед запуском перезапускаю RustFS

In [9]:
fresh_rustfs()
job("pairs-map", ["part", "full"], tuned, INSIDE, parts=8, parallel=8)
wait_pods("pairs-map", lambda phases: phases.count("Running") >= 3 and phases.count("Pending") >= 1)
sh("kubectl", "get", "pods", "-n", "hw5", "-l", "job-name=pairs-map", "-o", "wide")
pending = [pod["metadata"]["name"] for pod in pods("pairs-map") if pod["status"]["phase"] == "Pending"]
sh("kubectl", "get", "pod", pending[0], "-n", "hw5", "-o", "jsonpath={.status.conditions}")
print("итог задания:", wait_job("pairs-map"))
timeline("pairs-map")
save_logs("pairs-map")
save_events("pairs-map")

$ kubectl rollout restart deployment/rustfs -n hw5
deployment.apps/rustfs restarted



$ kubectl rollout status deployment/rustfs -n hw5 --timeout=300s
Waiting for deployment "rustfs" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "rustfs" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "rustfs" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "rustfs" rollout to finish: 0 of 1 updated replicas are available...
deployment "rustfs" successfully rolled out



$ kubectl apply -f k8s/jobs/pairs-map.yaml
job.batch/pairs-map created



$ kubectl get pods -n hw5 -l job-name=pairs-map -o wide
NAME                READY   STATUS    RESTARTS   AGE   IP          NODE             NOMINATED NODE   READINESS GATES
pairs-map-0-rxcrk   1/1     Running   0          4s    10.1.0.42   docker-desktop   <none>           <none>
pairs-map-1-2lwqs   1/1     Running   0          4s    10.1.0.44   docker-desktop   <none>           <none>
pairs-map-2-ps8fz   1/1     Running   0          4s    10.1.0.43   docker-desktop   <none>           <none>
pairs-map-3-dzbjl   0/1     Pending   0          4s    <none>      <none>           <none>           <none>
pairs-map-4-7tn2p   0/1     Pending   0          4s    <none>      <none>           <none>           <none>
pairs-map-5-r4chw   0/1     Pending   0          4s    <none>      <none>           <none>           <none>
pairs-map-6-6rxmk   0/1     Pending   0          4s    <none>      <none>           <none>           <none>
pairs-map-7-vbgpf   0/1     Pending   0          4s    <none>      <non

$ kubectl get pod pairs-map-3-dzbjl -n hw5 -o jsonpath={.status.conditions}
[{"lastProbeTime":null,"lastTransitionTime":"2026-11-23T19:39:50Z","message":"0/1 nodes are available: 1 Insufficient cpu. no new claims to deallocate, preemption: 0/1 nodes are available: 1 No preemption victims found for incoming pod.","observedGeneration":1,"reason":"Unschedulable","status":"False","type":"PodScheduled"}]



итог задания: Complete
под, часть, ждал планирования с, работал с, итог
pairs-map-0-rxcrk 0 0 126 Completed 0
pairs-map-1-2lwqs 1 0 129 Completed 0
pairs-map-2-ps8fz 2 0 130 Completed 0
pairs-map-3-dzbjl 3 135 110 Completed 0
pairs-map-4-7tn2p 4 131 111 Completed 0
pairs-map-5-r4chw 5 246 110 Completed 0
pairs-map-6-6rxmk 6 135 110 Completed 0
pairs-map-7-vbgpf 7 249 103 Completed 0


logs/pairs-map.log
[pod/pairs-map-3-dzbjl/pairs] часть 3 из 8 сплиты 141 - 187 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-map-3-dzbjl/pairs] готово сплитов 10 из 47
[pod/pairs-map-3-dzbjl/pairs] готово сплитов 20 из 47
[pod/pairs-map-3-dzbjl/pairs] готово сплитов 30 из 47
[pod/pairs-map-3-dzbjl/pairs] готово сплитов 40 из 47
[pod/pairs-map-3-dzbjl/pairs] готово сплитов 47 из 47
[pod/pairs-map-3-dzbjl/pairs] строк 1100503 вакансий Go 458 уникальных пар 3169 за 107.2 с, записал k8s/full/part-3.pickle
[pod/pairs-map-3-dzbjl/pairs] пик памяти контейнера 351 МиБ, предел 512 МиБ
[pod/pairs-map-4-7tn2p/pairs] часть 4 из 8 сплиты 188 - 234 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-map-4-7tn2p/pairs] готово сплитов 10 из 47
[pod/pairs-map-4-7tn2p/pairs] готово сплитов 20 из 47
[pod/pairs-map-4-7tn2p/pairs] готово сплитов 30 из 47
[pod/pairs-map-4-7tn2p/pairs] готово сплитов 40 из 47
[pod/pairs-map-4-7tn2p/pairs] готово сплитов 47 из 47
[pod/pairs-map-4-7tn2p/pairs] строк 1077787 вакансий G

logs/pairs-map.events.log
19:39:50 Normal Scheduled Pod/pairs-map-0-rxcrk: Successfully assigned hw5/pairs-map-0-rxcrk to docker-desktop
19:39:50 Normal Scheduled Pod/pairs-map-1-2lwqs: Successfully assigned hw5/pairs-map-1-2lwqs to docker-desktop
19:39:50 Normal Scheduled Pod/pairs-map-2-ps8fz: Successfully assigned hw5/pairs-map-2-ps8fz to docker-desktop
19:39:50 Warning FailedScheduling Pod/pairs-map-4-7tn2p: 0/1 nodes are available: 1 Insufficient cpu. no new claims to deallocate, preemption: 0/1 nodes are available: 1 No preemption victims found for incoming pod.
19:39:50 Normal SuccessfulCreate Job/pairs-map: Created pod: pairs-map-0-rxcrk
19:39:50 Normal SuccessfulCreate Job/pairs-map: Created pod: pairs-map-2-ps8fz
19:39:50 Normal SuccessfulCreate Job/pairs-map: Created pod: pairs-map-1-2lwqs
19:39:50 Normal SuccessfulCreate Job/pairs-map: Created pod: pairs-map-6-6rxmk
19:39:50 Normal SuccessfulCreate Job/pairs-map: Created pod: pairs-map-5-r4chw
19:39:50 Normal SuccessfulCrea

Части 0-2 стартовали сразу, части 3, 4 и 6 прождали в Pending 131-135 с, части 5 и 7 246-249 с, получилось три волны. Внутри кластера часть считается за 103-130 с, хотя работают три части сразу, а через хост одна часть шла 179-184 с

Reduce складывает 8 частей и сравнивает итог с ДЗ 3

In [10]:
job("pairs-reduce", ["reduce", "full"], "1Gi", INSIDE, cpu="1")
print("итог задания:", wait_job("pairs-reduce"))
save_logs("pairs-reduce")

$ kubectl apply -f k8s/jobs/pairs-reduce.yaml
job.batch/pairs-reduce created



итог задания: Complete


logs/pairs-reduce.log
[pod/pairs-reduce-0-62sb5/pairs] частей 8 уникальных пар 10999 сумма частот 48945 записал k8s/full/pairs.csv
[pod/pairs-reduce-0-62sb5/pairs] топ-10 пар: здесь | ДЗ 3
[pod/pairs-reduce-0-62sb5/pairs] 1 go + postgresql 967 | go + postgresql 967
[pod/pairs-reduce-0-62sb5/pairs] 2 docker + go 602 | docker + go 602
[pod/pairs-reduce-0-62sb5/pairs] 3 go + sql 464 | go + sql 464
[pod/pairs-reduce-0-62sb5/pairs] 4 git + go 455 | git + go 455
[pod/pairs-reduce-0-62sb5/pairs] 5 go + linux 441 | go + linux 441
[pod/pairs-reduce-0-62sb5/pairs] 6 go + kafka 406 | go + kafka 406
[pod/pairs-reduce-0-62sb5/pairs] 7 docker + postgresql 398 | docker + postgresql 398
[pod/pairs-reduce-0-62sb5/pairs] 8 go + kubernetes 386 | go + kubernetes 386
[pod/pairs-reduce-0-62sb5/pairs] 9 go + redis 335 | go + redis 335
[pod/pairs-reduce-0-62sb5/pairs] 10 go + rabbitmq 320 | go + rabbitmq 320
[pod/pairs-reduce-0-62sb5/pairs] топ-10 совпадает с ДЗ 3: да
[pod/pairs-reduce-0-62sb5/pairs] все пары

Итог 10999 пар с суммой частот 48945, топ-10 и все пары совпали с ДЗ 3

В отчёте ДЗ 3 топ-10 пар был без go, он подразумевается в каждой вакансии варианта, ниже тот же топ из итога кластера и из ДЗ 3

In [11]:
def top_without_go(key: str) -> list[tuple[str, int]]:
    text = s3.get_object(Bucket=common.BUCKET, Key=key)["Body"].read().decode("utf-8")
    rows = list(csv.reader(io.StringIO(text)))[1:]

    return [(row[0] + " + " + row[1], int(row[2])) for row in rows if "go" not in row[:2]][:10]


cluster = top_without_go("k8s/full/pairs.csv")
hw3 = top_without_go("results/mapreduce/skills_k2.csv")

for rank in range(10):
    print(rank + 1, cluster[rank][0], cluster[rank][1], "|", hw3[rank][0], hw3[rank][1])

print("совпадает:", cluster == hw3)

1 docker + postgresql 398 | docker + postgresql 398
2 kafka + postgresql 287 | kafka + postgresql 287
3 git + postgresql 274 | git + postgresql 274
4 postgresql + sql 241 | postgresql + sql 241
5 kubernetes + postgresql 234 | kubernetes + postgresql 234
6 docker + git 232 | docker + git 232
7 postgresql + redis 221 | postgresql + redis 221
8 postgresql + rabbitmq 220 | postgresql + rabbitmq 220
9 docker + kubernetes 214 | docker + kubernetes 214
10 git + sql 186 | git + sql 186
совпадает: True


## 6. Приоритеты и вытеснение

Два класса приоритета, hw5-low со значением 1000 и hw5-high со значением 100000. Фоновое задание из 3 частей с низким приоритетом занимает 12 ядер, когда все его части работают, запускаю срочное задание из одной части с высоким приоритетом. Свободного процессора для него нет, планировщик вытесняет одну фоновую часть, а Job заводит для неё новый под, который ждёт свободного процессора

In [12]:
fresh_rustfs()
sh("kubectl", "apply", "-f", "k8s/priority.yaml")
sh("kubectl", "get", "priorityclass")
job("pairs-low", ["part", "low"], tuned, INSIDE, parts=3, parallel=3, priority="hw5-low")
wait_pods("pairs-low", lambda phases: phases.count("Running") == 3)
time.sleep(20)
job("pairs-high", ["part", "high"], tuned, INSIDE, priority="hw5-high")
wait_pods("pairs-high", lambda phases: "Running" in phases)
sh("kubectl", "get", "pods", "-n", "hw5", "-l", "job-name in (pairs-low,pairs-high)", "-o", "wide")
print("срочное задание:", wait_job("pairs-high"))
print("фоновое задание:", wait_job("pairs-low"))
timeline("pairs-high")
timeline("pairs-low")
save_events("pairs-low")
save_events("pairs-high")
save_logs("pairs-high")
save_logs("pairs-low")

$ kubectl rollout restart deployment/rustfs -n hw5
deployment.apps/rustfs restarted



$ kubectl rollout status deployment/rustfs -n hw5 --timeout=300s
Waiting for deployment "rustfs" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "rustfs" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "rustfs" rollout to finish: 0 out of 1 new replicas have been updated...
Waiting for deployment "rustfs" rollout to finish: 0 of 1 updated replicas are available...
deployment "rustfs" successfully rolled out



$ kubectl apply -f k8s/priority.yaml
priorityclass.scheduling.k8s.io/hw5-low created
priorityclass.scheduling.k8s.io/hw5-high created



$ kubectl get priorityclass
NAME                      VALUE        GLOBAL-DEFAULT   AGE     PREEMPTIONPOLICY
hw5-high                  100000       false            1s      PreemptLowerPriority
hw5-low                   1000         false            1s      PreemptLowerPriority
system-cluster-critical   2000000000   false            7h25m   PreemptLowerPriority
system-node-critical      2000001000   false            7h25m   PreemptLowerPriority



$ kubectl apply -f k8s/jobs/pairs-low.yaml
job.batch/pairs-low created



$ kubectl apply -f k8s/jobs/pairs-high.yaml
job.batch/pairs-high created



... (4 outputs left out)

logs/pairs-low.events.log
19:46:46 Normal Scheduled Pod/pairs-low-0-4s59q: Successfully assigned hw5/pairs-low-0-4s59q to docker-desktop
19:46:46 Normal Scheduled Pod/pairs-low-1-6k2jv: Successfully assigned hw5/pairs-low-1-6k2jv to docker-desktop
19:46:46 Normal Scheduled Pod/pairs-low-2-r7545: Successfully assigned hw5/pairs-low-2-r7545 to docker-desktop
19:46:46 Normal SuccessfulCreate Job/pairs-low: Created pod: pairs-low-0-4s59q
19:46:46 Normal SuccessfulCreate Job/pairs-low: Created pod: pairs-low-2-r7545
19:46:46 Normal SuccessfulCreate Job/pairs-low: Created pod: pairs-low-1-6k2jv
19:46:47 Normal Pulled Pod/pairs-low-0-4s59q: Container image "pairs-job:hw5" already present on machine
19:46:47 Normal Pulled Pod/pairs-low-1-6k2jv: Container image "pairs-job:hw5" already present on machine
19:46:47 Normal Pulled Pod/pairs-low-2-r7545: Container image "pairs-job:hw5" already present on machine
19:46:48 Normal Created Pod/pairs-low-0-4s59q: Created container: pairs
19:46:48 Normal S

logs/pairs-high.events.log
19:47:11 Warning FailedScheduling Pod/pairs-high-0-sq4f7: 0/1 nodes are available: 1 Insufficient cpu.
19:47:11 Normal SuccessfulCreate Job/pairs-high: Created pod: pairs-high-0-sq4f7
19:47:42 Normal Scheduled Pod/pairs-high-0-sq4f7: Successfully assigned hw5/pairs-high-0-sq4f7 to docker-desktop
19:47:43 Normal Pulled Pod/pairs-high-0-sq4f7: Container image "pairs-job:hw5" already present on machine
19:47:43 Normal Created Pod/pairs-high-0-sq4f7: Created container: pairs
19:47:43 Normal Started Pod/pairs-high-0-sq4f7: Started container pairs
19:49:42 Normal Completed Job/pairs-high: Job completed



logs/pairs-high.log
[pod/pairs-high-0-sq4f7/pairs] часть 0 из 8 сплиты 0 - 46 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-high-0-sq4f7/pairs] готово сплитов 10 из 47
[pod/pairs-high-0-sq4f7/pairs] готово сплитов 20 из 47
[pod/pairs-high-0-sq4f7/pairs] готово сплитов 30 из 47
[pod/pairs-high-0-sq4f7/pairs] готово сплитов 40 из 47
[pod/pairs-high-0-sq4f7/pairs] готово сплитов 47 из 47
[pod/pairs-high-0-sq4f7/pairs] строк 1218176 вакансий Go 421 уникальных пар 2428 за 113.8 с, записал k8s/high/part-0.pickle
[pod/pairs-high-0-sq4f7/pairs] пик памяти контейнера 363 МиБ, предел 512 МиБ



logs/pairs-low.log
[pod/pairs-low-2-r7545/pairs] часть 2 из 8 сплиты 94 - 140 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-low-2-r7545/pairs] готово сплитов 10 из 47
[pod/pairs-low-2-r7545/pairs] готово сплитов 20 из 47
[pod/pairs-low-2-r7545/pairs] готово сплитов 30 из 47
[pod/pairs-low-2-r7545/pairs] готово сплитов 40 из 47
[pod/pairs-low-2-r7545/pairs] готово сплитов 47 из 47
[pod/pairs-low-2-r7545/pairs] строк 1185793 вакансий Go 607 уникальных пар 3474 за 131.4 с, записал k8s/low/part-2.pickle
[pod/pairs-low-2-r7545/pairs] пик памяти контейнера 352 МиБ, предел 512 МиБ
[pod/pairs-low-0-4s59q/pairs] часть 0 из 8 сплиты 0 - 46 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-low-0-4s59q/pairs] готово сплитов 10 из 47
[pod/pairs-low-0-4s59q/pairs] готово сплитов 20 из 47
[pod/pairs-low-0-4s59q/pairs] готово сплитов 30 из 47
[pod/pairs-low-0-4s59q/pairs] готово сплитов 40 из 47
[pod/pairs-low-0-4s59q/pairs] готово сплитов 47 из 47
[pod/pairs-low-0-4s59q/pairs] строк 1218176 вакансий Go 421

Срочный под вытеснил фоновую часть 1, в событиях это Preempted, и сам ждал 31 с, вытесненному поду дают 30 с на остановку, а python в роли PID 1 не обрабатывает SIGTERM. Новый под части 1 прождал в Pending 100 с, пока не закончились две другие фоновые части

## 7. Квота пространства имён

Отдельное пространство имён hw5-quota с ResourceQuota, не больше 8 ядер и 2 ГиБ памяти, подов не больше 4. Задание из 4 частей по 4 ядра, все 4 разрешено запускать сразу

In [13]:
sh("kubectl", "apply", "-f", "k8s/quota.yaml")
job("pairs-quota", ["part", "quota"], tuned, INSIDE, parts=4, parallel=4, namespace="hw5-quota")
wait_pods("pairs-quota", lambda phases: phases.count("Running") == 2, namespace="hw5-quota")
time.sleep(5)
sh("kubectl", "get", "pods", "-n", "hw5-quota")
sh("kubectl", "describe", "resourcequota", "hw5-quota", "-n", "hw5-quota")
print("итог задания:", wait_job("pairs-quota", namespace="hw5-quota"))
timeline("pairs-quota", namespace="hw5-quota")
save_events("pairs-quota", namespace="hw5-quota")
save_logs("pairs-quota", namespace="hw5-quota")
sh("kubectl", "describe", "resourcequota", "hw5-quota", "-n", "hw5-quota")

$ kubectl apply -f k8s/quota.yaml
namespace/hw5-quota created
resourcequota/hw5-quota created



$ kubectl apply -f k8s/jobs/pairs-quota.yaml
job.batch/pairs-quota created



$ kubectl get pods -n hw5-quota
NAME                  READY   STATUS    RESTARTS   AGE
pairs-quota-0-ml2ft   1/1     Running   0          8s
pairs-quota-2-qvntw   1/1     Running   0          8s

$ kubectl describe resourcequota hw5-quota -n hw5-quota
Name:            hw5-quota
Namespace:       hw5-quota
Resource         Used  Hard
--------         ----  ----
limits.memory    1Gi   2Gi
pods             2     4
requests.cpu     8     8
requests.memory  1Gi   2Gi



итог задания: Complete
под, часть, ждал планирования с, работал с, итог


pairs-quota-0-ml2ft 0 0 97 Completed 0
pairs-quota-2-qvntw 2 0 95 Completed 0
pairs-quota-1-bk5rf 1 0 79 Completed 0
pairs-quota-3-h4h8f 3 0 79 Completed 0


logs/pairs-quota.events.log
19:50:22 Normal Scheduled Pod/pairs-quota-0-ml2ft: Successfully assigned hw5-quota/pairs-quota-0-ml2ft to docker-desktop
19:50:22 Normal Scheduled Pod/pairs-quota-2-qvntw: Successfully assigned hw5-quota/pairs-quota-2-qvntw to docker-desktop
19:50:22 Normal SuccessfulCreate Job/pairs-quota: Created pod: pairs-quota-0-ml2ft
19:50:22 Warning FailedCreate Job/pairs-quota: Error creating: pods "pairs-quota-1-4br7j" is forbidden: exceeded quota: hw5-quota, requested: requests.cpu=4, used: requests.cpu=8, limited: requests.cpu=8
19:50:22 Normal SuccessfulCreate Job/pairs-quota: Created pod: pairs-quota-2-qvntw
19:50:23 Normal Pulled Pod/pairs-quota-0-ml2ft: Container image "pairs-job:hw5" already present on machine
19:50:23 Normal Created Pod/pairs-quota-0-ml2ft: Created container: pairs
19:50:23 Normal Started Pod/pairs-quota-0-ml2ft: Started container pairs
19:50:23 Normal Pulled Pod/pairs-quota-2-qvntw: Container image "pairs-job:hw5" already present on machine

logs/pairs-quota.log
[pod/pairs-quota-0-ml2ft/pairs] часть 0 из 8 сплиты 0 - 46 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-quota-0-ml2ft/pairs] готово сплитов 10 из 47
[pod/pairs-quota-0-ml2ft/pairs] готово сплитов 20 из 47
[pod/pairs-quota-0-ml2ft/pairs] готово сплитов 30 из 47
[pod/pairs-quota-0-ml2ft/pairs] готово сплитов 40 из 47
[pod/pairs-quota-0-ml2ft/pairs] готово сплитов 47 из 47
[pod/pairs-quota-0-ml2ft/pairs] строк 1218176 вакансий Go 421 уникальных пар 2428 за 95.0 с, записал k8s/quota/part-0.pickle
[pod/pairs-quota-0-ml2ft/pairs] пик памяти контейнера 352 МиБ, предел 512 МиБ
[pod/pairs-quota-1-bk5rf/pairs] часть 1 из 8 сплиты 47 - 93 воркеров 4 S3 rustfs.hw5.svc:9000
[pod/pairs-quota-1-bk5rf/pairs] готово сплитов 10 из 47
[pod/pairs-quota-1-bk5rf/pairs] готово сплитов 20 из 47
[pod/pairs-quota-1-bk5rf/pairs] готово сплитов 30 из 47
[pod/pairs-quota-1-bk5rf/pairs] готово сплитов 40 из 47
[pod/pairs-quota-1-bk5rf/pairs] готово сплитов 47 из 47
[pod/pairs-quota-1-bk5rf/pair

Квота пустила 2 пода по 4 ядра, занято 8 из 8, остальные части Job не смог создать, в событиях FailedCreate exceeded quota. По ядрам узел принял бы и третий под, поэтому части не висят в Pending, как в разделе 5, а не появляются, пока квота не освободится

In [14]:
sh(sys.executable, "-m", "nbconvert", "--to", "html", "kubernetes.ipynb")
PRINT_STYLE = """<style>
@page { size: A4 landscape; margin: 8mm; }
@media print {
  .jp-InputPrompt, .jp-OutputPrompt { display: none !important; }
  .jp-OutputArea-output, .jp-OutputArea-child, .jp-Cell-outputWrapper, .jp-Cell-inputWrapper, .jp-InputArea-editor { overflow: visible !important; }
  .jp-RenderedHTMLCommon table { font-size: 8px !important; table-layout: auto !important; }
  .jp-RenderedHTMLCommon td, .jp-RenderedHTMLCommon th { padding: 1px 3px !important; }
  .jp-RenderedText pre, .jp-InputArea-editor pre, .highlight pre { font-size: 8px !important; line-height: 1.25 !important; white-space: pre-wrap !important; word-break: break-all !important; }
  .jp-RenderedImage img { max-width: 100% !important; }
}
</style>
"""
report = PROJECT / "kubernetes.html"
report.write_text(report.read_text(encoding="utf-8").replace("</head>", PRINT_STYLE + "</head>", 1), encoding="utf-8")
print("html", report.stat().st_size, "байт")

$ C:\python\Study\ITMO_3nd\EngData\Kubernetes\.venv\Scripts\python.exe -m nbconvert --to html kubernetes.ipynb
[NbConvertApp] Converting notebook kubernetes.ipynb to html
[NbConvertApp] Writing 363846 bytes to kubernetes.html



html 374966 байт
